Jeg modellerer:
2 populationer:
 - main population (observeret)
 - ghost population (ukendt)
lineages bevæger sig bagud i tid

In [4]:
from phasic import Graph
import numpy as np

def ghost_coalescent_model(state, 
                           coal_rate_main=1.0, 
                           coal_rate_ghost=1.0,
                           mig_rate=0.05):
    
    k_main, k_ghost = state
    transitions = []

    # --- COALESCENCE MAIN ---
    if k_main > 1:
        child = state.copy()
        child[0] -= 1
        rate = coal_rate_main * (k_main * (k_main - 1) / 2)
        transitions.append([child, rate])

    # --- COALESCENCE GHOST ---
    if k_ghost > 1:
        child = state.copy()
        child[1] -= 1
        rate = coal_rate_ghost * (k_ghost * (k_ghost - 1) / 2)
        transitions.append([child, rate])

    # --- MIGRATION MAIN → GHOST ---
    if k_main > 0:
        child = state.copy()
        child[0] -= 1
        child[1] += 1
        transitions.append([child, k_main * mig_rate])

    # --- MIGRATION GHOST → MAIN ---
    if k_ghost > 0:
        child = state.copy()
        child[1] -= 1
        child[0] += 1
        transitions.append([child, k_ghost * mig_rate])

    return transitions

In [ ]:
graph = Graph(
    ghost_coalescent_model,
    ipv=[4, 0],   # 4 lineages i main, 0 i ghost
    coal_rate_main=1.0,
    coal_rate_ghost=0.5,
    mig_rate=0.05
)

In [ ]:
graph.plot()
graph.expectation()
graph.sample(1000)

# Kobling til Baboon data

In [ ]:
ds = sg.load_dataset("/faststorage/project/baboondiversity/data/PG_panu3_phased_chromosomes_4_7_2021_ZARR/chr20.phased.rehead.vcz")
ds

In [ ]:
positions = ds.variant_position.values
observations = np.diff(positions)

In [ ]:
genotypes = ds.call_genotype.values

h1 = genotypes[:, 0, 0]
h2 = genotypes[:, 1, 0]

same = (h1 == h2)

segments = []
length = 0

for val in same:
    if val:
        length += 1
    else:
        if length > 0:
            segments.append(length)
        length = 0

observations = np.array(segments)